# 🚀 Notebook 4 — Optimisation & Déploiement du Modèle Final
---
**Objectif** : Optimiser le meilleur modèle identifié au Notebook 3 via GridSearchCV et RandomizedSearchCV, diagnostiquer le surapprentissage avec les learning curves, évaluer les performances finales et déployer le modèle dans l'API Django.

## Table des matières
1. Chargement et rappel des performances
2. GridSearchCV — exploration exhaustive
3. RandomizedSearchCV — exploration élargie
4. Comparaison et sélection de la meilleure configuration
5. Learning curves — diagnostic du surapprentissage
6. Évaluation finale complète
7. Tests de prédiction sur des cas réels
8. Sauvegarde et déploiement Django


## 1. Chargement et rappel des performances

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import warnings, json, joblib, os, time
from scipy.stats import randint, uniform
warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"]    = (13, 5)
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False

from sklearn.model_selection import (train_test_split, GridSearchCV,
                                      RandomizedSearchCV, learning_curve, KFold)
from sklearn.preprocessing   import StandardScaler, TargetEncoder
from sklearn.impute          import SimpleImputer
from sklearn.pipeline        import Pipeline
from sklearn.compose         import ColumnTransformer
from sklearn.metrics         import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble        import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model    import Ridge

try:
    from xgboost import XGBRegressor
    XGBOOST_OK = True
except ImportError:
    XGBOOST_OK = False

print("Librairies chargees.")


In [ ]:
df = pd.read_csv("dataset_final.csv")
df = df.dropna(subset=["price"])

with open("features_config.json")    as f: cfg  = json.load(f)
with open("comparison_results.json") as f: comp = json.load(f)

NUMERIC_FEATURES     = cfg["NUMERIC_FEATURES"]
CATEGORICAL_FEATURES = cfg["CATEGORICAL_FEATURES"]
TARGET               = cfg["TARGET"]
best_name            = comp.get("best_model", "Random Forest")

X     = df[[c for c in NUMERIC_FEATURES + CATEGORICAL_FEATURES if c in df.columns]]
y     = df[TARGET]
y_log = np.log1p(y)

num_feats = [f for f in NUMERIC_FEATURES     if f in X.columns]
cat_feats = [f for f in CATEGORICAL_FEATURES if f in X.columns]

X_train, X_test,  y_train,  y_test   = train_test_split(X, y,     test_size=0.2, random_state=42)
_,       _,       yl_train, yl_test   = train_test_split(X, y_log, test_size=0.2, random_state=42)

def build_pipeline(model):
    np_ = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
    cp_ = Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="inconnu")),
                    ("enc", TargetEncoder(target_type="continuous", smooth="auto"))])
    prep = ColumnTransformer([("num", np_, num_feats), ("cat", cp_, cat_feats)])
    return Pipeline([("prep", prep), ("model", model)])

print(f"Meilleur modele (NB3)   : {best_name}")
print(f"R2 avant optimisation   : {comp.get(best_name, {}).get('R2 test', 'N/A')}")
print(f"Train : {len(X_train):,}  |  Test : {len(X_test):,}")


## 2. GridSearchCV — exploration exhaustive

In [ ]:
GRIDS = {
    "Random Forest": {
        "model__n_estimators":      [100, 200, 300, 500],
        "model__max_depth":         [None, 10, 20, 30],
        "model__min_samples_split": [2, 5, 10],
        "model__min_samples_leaf":  [1, 2, 4],
        "model__max_features":      ["sqrt", "log2"],
    },
    "Gradient Boosting": {
        "model__n_estimators":      [200, 300, 500],
        "model__learning_rate":     [0.03, 0.05, 0.1],
        "model__max_depth":         [3, 5, 7],
        "model__subsample":         [0.7, 0.8, 1.0],
    },
    "XGBoost": {
        "model__n_estimators":      [200, 300, 500],
        "model__learning_rate":     [0.03, 0.05, 0.1],
        "model__max_depth":         [4, 6, 8],
        "model__subsample":         [0.7, 0.8, 1.0],
        "model__colsample_bytree":  [0.7, 0.8, 1.0],
        "model__reg_alpha":         [0, 0.1, 0.5],
    },
    "Ridge": {"model__alpha": [0.1, 1, 10, 100, 1000, 5000]},
}
MODELS_MAP = {
    "Random Forest":    RandomForestRegressor(random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "Ridge":            Ridge(),
}
if XGBOOST_OK:
    MODELS_MAP["XGBoost"] = XGBRegressor(random_state=42, verbosity=0, n_jobs=-1)

key        = best_name if best_name in MODELS_MAP else "Random Forest"
model_base = MODELS_MAP[key]
param_grid = GRIDS.get(key, GRIDS["Random Forest"])

n_combs = 1
for v in param_grid.values(): n_combs *= len(v)
print(f"Modele          : {key}")
print(f"Combinaisons    : {n_combs:,} x 5-fold = {n_combs*5:,} fits")
print("GridSearchCV en cours...")

t0 = time.time()
gs = GridSearchCV(build_pipeline(model_base), param_grid,
                  cv=5, scoring="r2", n_jobs=-1, verbose=1,
                  refit=True, return_train_score=True)
gs.fit(X_train, yl_train)
print(f"
Termine en {time.time()-t0:.1f} s")
print(f"Meilleur CV R2  : {gs.best_score_:.4f}")
print("Meilleurs parametres :")
for k, v in gs.best_params_.items():
    print(f"  {k:<45}: {v}")

yp_gs  = np.expm1(gs.best_estimator_.predict(X_test))
r2_gs  = r2_score(y_test, yp_gs)
mae_gs = mean_absolute_error(y_test, yp_gs)
print(f"
R2 test (GridSearch)  : {r2_gs:.4f}")
print(f"MAE test (GridSearch) : {mae_gs:,.0f} FCFA")


## 3. RandomizedSearchCV — exploration élargie

In [ ]:
RAND_GRIDS = {
    "Random Forest": {
        "model__n_estimators":      randint(100, 800),
        "model__max_depth":         [None, 5, 10, 15, 20, 25, 30, 40],
        "model__min_samples_split": randint(2, 15),
        "model__min_samples_leaf":  randint(1, 8),
        "model__max_features":      ["sqrt", "log2", 0.3, 0.5, 0.7],
        "model__bootstrap":         [True, False],
    },
    "Gradient Boosting": {
        "model__n_estimators":      randint(100, 600),
        "model__learning_rate":     uniform(0.01, 0.25),
        "model__max_depth":         randint(2, 10),
        "model__subsample":         uniform(0.5, 0.5),
        "model__min_samples_split": randint(2, 12),
    },
    "XGBoost": {
        "model__n_estimators":      randint(100, 700),
        "model__learning_rate":     uniform(0.01, 0.2),
        "model__max_depth":         randint(3, 12),
        "model__subsample":         uniform(0.5, 0.5),
        "model__colsample_bytree":  uniform(0.5, 0.5),
        "model__reg_alpha":         uniform(0, 2),
        "model__reg_lambda":        uniform(0.5, 3),
    },
}
N_ITER    = 150
rand_grid = RAND_GRIDS.get(key, RAND_GRIDS["Random Forest"])
print(f"RandomizedSearchCV - {N_ITER} iterations x 5-fold = {N_ITER*5:,} fits")
print("En cours...")

t0 = time.time()
rs = RandomizedSearchCV(build_pipeline(MODELS_MAP[key]), rand_grid,
                        n_iter=N_ITER, cv=5, scoring="r2", n_jobs=-1,
                        verbose=1, random_state=42, refit=True, return_train_score=True)
rs.fit(X_train, yl_train)
print(f"
Termine en {time.time()-t0:.1f} s")
print(f"Meilleur CV R2  : {rs.best_score_:.4f}")
print("Meilleurs parametres :")
for k, v in rs.best_params_.items():
    print(f"  {k:<45}: {v}")

yp_rs  = np.expm1(rs.best_estimator_.predict(X_test))
r2_rs  = r2_score(y_test, yp_rs)
mae_rs = mean_absolute_error(y_test, yp_rs)
print(f"
R2 test (RandomizedSearch)  : {r2_rs:.4f}")
print(f"MAE test (RandomizedSearch) : {mae_rs:,.0f} FCFA")


## 4. Comparaison et sélection

In [ ]:
r2_before = float(comp.get(key, {}).get("R2 test", 0))

print("=" * 65)
print(f"  {'Methode':<30} {'R2 test':>10} {'MAE (FCFA)':>18}")
print("-" * 65)
print(f"  {'Avant optimisation':<30} {r2_before:>10.4f} {'N/A':>18}")
print(f"  {'GridSearchCV':<30} {r2_gs:>10.4f} {mae_gs:>18,.0f}")
print(f"  {'RandomizedSearchCV':<30} {r2_rs:>10.4f} {mae_rs:>18,.0f}")

if r2_rs >= r2_gs:
    best_opt    = rs.best_estimator_; best_method = "RandomizedSearchCV"
    best_params = rs.best_params_;    r2_opt = r2_rs; mae_opt = mae_rs; yp_opt = yp_rs
else:
    best_opt    = gs.best_estimator_; best_method = "GridSearchCV"
    best_params = gs.best_params_;    r2_opt = r2_gs; mae_opt = mae_gs; yp_opt = yp_gs

rmse_opt = np.sqrt(mean_squared_error(y_test, yp_opt))
print(f"
Methode retenue  : {best_method}")
print(f"R2 test          : {r2_opt:.4f}  (gain : {r2_opt - r2_before:+.4f})")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
gs_df = pd.DataFrame(gs.cv_results_)
rs_df = pd.DataFrame(rs.cv_results_)
axes[0].scatter(gs_df["mean_train_score"], gs_df["mean_test_score"],
                alpha=0.4, s=20, color="#2196F3", label="GridSearch")
axes[0].scatter(rs_df["mean_train_score"], rs_df["mean_test_score"],
                alpha=0.4, s=20, color="#FF9800", label="RandomSearch")
axes[0].plot([0,1],[0,1],"r--",lw=1)
axes[0].set_xlabel("CV R2 train"); axes[0].set_ylabel("CV R2 test")
axes[0].set_title("Train vs Test - toutes combinaisons", fontweight="bold")
axes[0].legend()

for label, scores, color in [("Grid", gs_df["mean_test_score"], "#2196F3"),
                               ("Rand", rs_df["mean_test_score"], "#FF9800")]:
    axes[1].hist(scores, bins=30, alpha=0.6, color=color, label=label, edgecolor="white")
axes[1].set_xlabel("CV R2 test")
axes[1].set_title("Distribution des scores de recherche", fontweight="bold")
axes[1].legend()
plt.tight_layout()
plt.savefig("fig_search_results.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. Learning curves — diagnostic du surapprentissage

In [ ]:
print("Calcul des learning curves...")
t0 = time.time()
tr_sizes, tr_sc, val_sc = learning_curve(
    best_opt, X_train, yl_train,
    train_sizes=np.linspace(0.1, 1.0, 12),
    cv=5, scoring="r2", n_jobs=-1)
print(f"Termine en {time.time()-t0:.1f} s")

tr_m, tr_s   = tr_sc.mean(1), tr_sc.std(1)
val_m, val_s = val_sc.mean(1), val_sc.std(1)
gap          = tr_m - val_m

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].plot(tr_sizes, tr_m,  "o-", color="#2196F3", label="Entrainement", lw=2)
axes[0].fill_between(tr_sizes, tr_m-tr_s, tr_m+tr_s, alpha=0.15, color="#2196F3")
axes[0].plot(tr_sizes, val_m, "s-", color="#4CAF50", label="Validation (CV)", lw=2)
axes[0].fill_between(tr_sizes, val_m-val_s, val_m+val_s, alpha=0.15, color="#4CAF50")
axes[0].set_xlabel("Taille d'entrainement"); axes[0].set_ylabel("R2")
axes[0].set_title(f"Learning Curves - {key} ({best_method})", fontweight="bold")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(tr_sizes, gap, "o-", color="#ef5350", lw=2)
axes[1].axhline(0.05, color="orange", linestyle="--", label="Seuil 0.05")
axes[1].axhline(0.10, color="red",    linestyle="--", label="Seuil 0.10")
axes[1].fill_between(tr_sizes, 0, gap, alpha=0.2, color="#ef5350")
axes[1].set_xlabel("Taille d'entrainement")
axes[1].set_ylabel("Gap R2 (train - validation)")
axes[1].set_title("Evolution du surapprentissage", fontweight="bold")
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("fig_learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()

gf   = gap[-1]
diag = ("Bonne generalisation" if gf < 0.05 else
        "leger surapprentissage - acceptable" if gf < 0.10 else
        "Surapprentissage detecte - regulariser davantage")
print(f"
Diagnostic : {diag}")
print(f"Gap final  : {gf:.4f}  |  Score val. final : {val_m[-1]:.4f}")


## 6. Évaluation finale complète

In [ ]:
resid   = y_test.values - yp_opt
rel_e   = np.abs(resid) / (y_test.values + 1) * 100
r2_f    = r2_score(y_test, yp_opt)
mae_f   = mean_absolute_error(y_test, yp_opt)
rmse_f  = np.sqrt(mean_squared_error(y_test, yp_opt))
mape_f  = np.mean(rel_e)
pct_20  = (rel_e < 20).mean() * 100
pct_30  = (rel_e < 30).mean() * 100

print("=" * 58)
print("  METRIQUES FINALES DU MODELE OPTIMISE")
print("=" * 58)
print(f"  R2                          : {r2_f:.4f}")
print(f"  MAE                         : {mae_f:>15,.0f} FCFA")
print(f"  RMSE                        : {rmse_f:>15,.0f} FCFA")
print(f"  MAPE                        : {mape_f:.2f} %")
print(f"  Erreur mediane              : {np.median(np.abs(resid)):>15,.0f} FCFA")
print(f"  Predictions a +-20 %        : {pct_20:.1f} %")
print(f"  Predictions a +-30 %        : {pct_30:.1f} %")
obj = "OBJECTIF ATTEINT" if r2_f >= 0.90 else f"manque {0.90 - r2_f:.4f}"
print(f"  Objectif R2 >= 0.90         : {obj}")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
lim = max(y_test.max(), yp_opt.max()) / 1e6

axes[0,0].scatter(y_test/1e6, yp_opt/1e6, alpha=0.3, s=8, color="#2196F3")
axes[0,0].plot([0,lim],[0,lim],"r--",lw=2,label="Prediction parfaite")
axes[0,0].set_xlabel("Prix reel (M FCFA)"); axes[0,0].set_ylabel("Prix predit (M FCFA)")
axes[0,0].set_title(f"Reel vs Predit - R2={r2_f:.4f}", fontweight="bold")
axes[0,0].legend()

axes[0,1].hist(resid/1e6, bins=60, color="#4CAF50", alpha=0.7, edgecolor="white")
axes[0,1].axvline(0, color="red", linestyle="--", lw=2)
axes[0,1].axvline(resid.mean()/1e6, color="orange", linestyle="--",
                   label=f"Moyenne = {resid.mean()/1e6:.1f}M")
axes[0,1].set_xlabel("Residu (M FCFA)")
axes[0,1].set_title("Distribution des residus", fontweight="bold")
axes[0,1].legend()

axes[1,0].scatter(yp_opt/1e6, resid/1e6, alpha=0.3, s=8, color="#9C27B0")
axes[1,0].axhline(0, color="red", linestyle="--", lw=1.5)
axes[1,0].set_xlabel("Predit (M FCFA)"); axes[1,0].set_ylabel("Residu (M FCFA)")
axes[1,0].set_title("Residus vs Predits", fontweight="bold")

rs_  = np.sort(rel_e)
pct_ = np.arange(1, len(rs_)+1) / len(rs_) * 100
axes[1,1].plot(rs_, pct_, "-", color="#2196F3", lw=2)
axes[1,1].axvline(20, color="green",  linestyle="--", label=f"{pct_20:.0f} % a +-20 %")
axes[1,1].axvline(30, color="orange", linestyle="--", label=f"{pct_30:.0f} % a +-30 %")
axes[1,1].set_xlabel("Erreur relative (%)"); axes[1,1].set_ylabel("% predictions")
axes[1,1].set_title("Courbe cumulative des erreurs relatives", fontweight="bold")
axes[1,1].set_xlim(0, 100); axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

plt.suptitle(f"Evaluation finale - {key} ({best_method})", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("fig_evaluation_finale.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Tests de prédiction sur des cas réels

In [ ]:
from math import radians, sin, cos, sqrt, atan2

CITY_COORDS = {
    "Almadies":(14.7453,-17.5109), "Ngor":(14.7490,-17.5140),
    "Ouakam":  (14.7237,-17.4942), "Plateau":(14.6928,-17.4467),
    "Pikine":  (14.7546,-17.3947), "Fann":(14.6961,-17.4603),
    "Mermoz":  (14.7100,-17.4750), "Rufisque":(14.7156,-17.2736),
    "Yoff":    (14.7575,-17.4900), "Grand Yoff":(14.7500,-17.4600),
}
POI_CENTRE  = (14.6928,-17.4467)
POI_MER     = [(14.7457,-17.5197),(14.7247,-17.5025)]
POI_AER     = (14.7397,-17.4902)
POI_PARC    = (14.7122,-17.4488)
POI_UCAD    = (14.6925,-17.4636)
POI_VDN     = (14.7200,-17.4600)
PREMIUM     = ["Almadies","Ngor","Ouakam","Mermoz","Fann","Plateau"]

def hav(la1,lo1,la2,lo2):
    R=6371; la1,lo1,la2,lo2=map(radians,[la1,lo1,la2,lo2])
    a=sin((la2-la1)/2)**2+cos(la1)*cos(la2)*sin((lo2-lo1)/2)**2
    return R*2*atan2(sqrt(a),sqrt(1-a))

def predict_bien(surface_area, bedrooms, bathrooms, city, property_type, source="coinafrique"):
    lat, lon = CITY_COORDS.get(city, (14.6928,-17.4467))
    dm  = min(hav(lat,lon,*p) for p in POI_MER)
    cmp = float(df[df["city_mean_price"].notna()]["city_mean_price"].mean())

    row = {
        "surface_area":surface_area, "bedrooms":bedrooms, "bathrooms":bathrooms,
        "dist_mer":dm, "dist_centre":hav(lat,lon,*POI_CENTRE),
        "dist_aeroport":hav(lat,lon,*POI_AER), "dist_parc":hav(lat,lon,*POI_PARC),
        "dist_ucad":hav(lat,lon,*POI_UCAD),    "dist_vdn":hav(lat,lon,*POI_VDN),
        "rooms_total":(bedrooms or 0)+(bathrooms or 0),
        "is_premium":int(city in PREMIUM),
        "surface_per_room":surface_area/bedrooms if bedrooms and bedrooms>0 else surface_area,
        "log_surface":np.log1p(surface_area),
        "log_dist_mer":np.log1p(dm),
        "bath_bed_ratio":bathrooms/bedrooms if bedrooms and bedrooms>0 else 0,
        "city_mean_price":cmp, "city_median_price":cmp*0.9, "city_count":50,
        "has_piscine":0,"has_standing":0,"has_meuble":0,"has_parking":0,
        "has_ascenseur":0,"has_gardiennage":0,"has_balcon":0,"has_vue_mer":0,
        "has_groupe_elec":0,"has_climatise":0,
        "property_type":property_type, "source":source,
    }
    Xp    = pd.DataFrame([row])
    cols  = [c for c in num_feats + cat_feats if c in Xp.columns]
    price = max(0, np.expm1(best_opt.predict(Xp[cols])[0]))
    margin = price * 0.15
    return {
        "Prix estime":  f"{price:,.0f} FCFA",
        "Fourchette":   f"{max(0,price-margin):,.0f} - {price+margin:,.0f} FCFA",
        "En millions":  f"{price/1e6:.2f} M FCFA",
    }

cas = [
    dict(label="Villa 6 pieces - Almadies (luxe bord de mer)",
         surface_area=350, bedrooms=5, bathrooms=4, city="Almadies", property_type="Villa"),
    dict(label="Appartement F3 - Ouakam (milieu de gamme)",
         surface_area=100, bedrooms=3, bathrooms=2, city="Ouakam",   property_type="Appartement"),
    dict(label="Appartement F2 - Pikine (populaire)",
         surface_area=60,  bedrooms=2, bathrooms=1, city="Pikine",   property_type="Appartement"),
    dict(label="Terrain 500 m2 - Ngor (bord de mer)",
         surface_area=500, bedrooms=1, bathrooms=1, city="Ngor",     property_type="Terrain"),
    dict(label="Maison F5 - Rufisque (peripherie)",
         surface_area=200, bedrooms=4, bathrooms=3, city="Rufisque", property_type="Villa"),
]

print("=" * 65)
print("  TESTS DE PREDICTION SUR DES CAS REELS")
print("=" * 65)
for cas_ in cas:
    label = cas_.pop("label")
    res   = predict_bien(**cas_)
    print(f"
  {label}")
    for k, v in res.items():
        print(f"    {k:<18} : {v}")


## 8. Sauvegarde et déploiement Django

In [ ]:
MODEL_DIR    = os.path.join("..", "properties", "ml")
MODEL_PATH   = os.path.join(MODEL_DIR, "model.pkl")
RESULTS_PATH = os.path.join(MODEL_DIR, "results.json")
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump({
    "pipeline":             best_opt,
    "best_model_name":      key,
    "search_method":        best_method,
    "best_params":          {str(k): str(v) for k, v in best_params.items()},
    "features":             num_feats + cat_feats,
    "numeric_features":     num_feats,
    "categorical_features": cat_feats,
    "metrics": {
        "r2":     round(r2_f, 4), "mae":  round(mae_f, 2),
        "rmse":   round(rmse_f, 2), "mape": round(mape_f, 2),
        "pct_20": round(pct_20, 1), "pct_30": round(pct_30, 1),
    },
}, MODEL_PATH)
print(f"Modele sauvegarde   -> {MODEL_PATH}")

final = {
    "best_model":    key,
    "search_method": best_method,
    "metrics": {
        "R2": round(r2_f,4), "MAE": round(mae_f,2),
        "RMSE": round(rmse_f,2), "MAPE (%)": round(mape_f,2),
        "pct_20": round(pct_20,1), "pct_30": round(pct_30,1),
    },
    "best_params": {str(k): str(v) for k, v in best_params.items()},
}
with open(RESULTS_PATH, "w") as f:
    json.dump(final, f, indent=2, default=str)
print(f"Resultats sauvegardes -> {RESULTS_PATH}")

print("""
RECAPITULATIF COMPLET DU PIPELINE ML
======================================
NB1 : EDA - 5 sources, distributions, correlations
       Separation vente/location sur donnees textuelles reelles
NB2 : Preprocessing - 30 features, Target Encoding, filtrage homogene
NB3 : 8 modeles compares avec log(prix) + TargetEncoder
NB4 : GridSearch + RandomizedSearch (150 iter) + learning curves

DEPLOIEMENT API DJANGO
  POST /api/properties/predict/    -> prediction de prix
  GET  /api/properties/ml/results/ -> metriques du modele
""")
